In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from sklearn.metrics import accuracy_score, classification_report


IMAGE_SIZE = 224  

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = len(test_ds.classes)  
model_mvit = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=False,
    num_classes=num_classes
)

model_mvit.load_state_dict(torch.load("models/swin.pth", map_location=device))
model_mvit = model_mvit.to(device)
model_mvit.eval()


predictionresult = []

def inference(model1, loader):
    y_true, y_pred = [], []
    class_names = loader.dataset.classes  
    global_index = 0  

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            out1 = model1(imgs)
            prob1 = F.softmax(out1, dim=1)
            preds = prob1.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred


y_true, y_pred = inference(
    model_mvit,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))



Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7197b73d0cd0>>
Traceback (most recent call last):
  File "/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5060 which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 13.0 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_war

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
